In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('..')
from pathlib import Path
from flowrec.losses import relative_error

In [ ]:
data_path_original1 = Path("/storage0/ym917/data/simulations/kolsol/dim3_re34_k32_f4_dt01_grid64_478_t200-250.h5")
data_path_original2 = Path("/storage0/ym917/data/simulations/kolsol/dim3_re34_k32_f4_dt01_grid64_478_t280-330.h5")
data_path_perturb = Path("/storage0/ym917/data/simulations/kolsol/perturbation/dim3_re34_k32_f4_dt01_grid64_478_t200-330.h5")
data_path_perturb.exists()

In [ ]:
with h5py.File(data_path_original1) as hf:
    print(float(hf.get('dt')[()]))
    print(float(hf.get('re')[()]))
    data200 = np.array(hf.get('state'))
    print(data200.shape)
with h5py.File(data_path_original2) as hf:
    # print(float(hf.get('dt')[()]))
    # print(float(hf.get('re')[()]))
    data280 = np.array(hf.get('state'))
with h5py.File(data_path_perturb) as hf:
    print(float(hf.get('dt')[()]))
    print(float(hf.get('re')[()]))
    _data = np.array(hf.get('state'))
    print(_data.shape)

## Diverging norm from perturbation

In [ ]:
data200new = _data[:500,...]
data280new = _data[800:,...]

In [ ]:
def normdiff(data1, data2):
    diff = data2 - data1
    diff_norm = np.sqrt(np.einsum('txyzu -> t', diff**2))
    return diff_norm/diff_norm[0]

In [ ]:
diff200_norm = normdiff(data200, data200new)
diff280_norm = normdiff(data280, data280new)

In [ ]:
fig, axes = plt.subplots(1,2, sharey=True, figsize=(5,3))
axes[0].semilogy(np.arange(0,50,0.1), diff200_norm)
axes[0].set(xlabel='t', ylabel='Norm of difference', )
axes[1].semilogy(np.arange(80,80+50,0.1), diff280_norm)
axes[1].set(xlabel='t')
plt.show()

In [ ]:
with h5py.File("../local_data/kolmogorov/perturbation/seed1372_t150-200.h5") as hf:
    data1_1372 = np.array(hf.get('state'))
with h5py.File("../local_data/kolmogorov/perturbation/seed1372_t150-200_uniform-e-0001.h5") as hf:
    data2_1372 = np.array(hf.get('state'))
diff_1372_norm = normdiff(data1_1372, data2_1372)

In [ ]:
with h5py.File("../local_data/kolmogorov/perturbation/seed691_t150-200.h5") as hf:
    data1_691 = np.array(hf.get('state'))
with h5py.File("../local_data/kolmogorov/perturbation/seed691_t150-200_uniform-e-0001.h5") as hf:
    data2_691 = np.array(hf.get('state'))
diff_691_norm = normdiff(data1_691, data2_691)

In [ ]:
with h5py.File("../local_data/kolmogorov/perturbation/seed478_t200-250_uniform-e-0001.h5") as hf:
    data2_478 = np.array(hf.get('state'))
diff_478_norm = normdiff(data200, data2_478)

In [ ]:
fig, ax = plt.subplots(1, sharey=True, figsize=(4,4))
ax.semilogy(np.arange(0,50,0.1), diff_1372_norm, 'k', alpha=0.7)
ax.semilogy(np.arange(0,50,0.1), diff_691_norm, 'k', alpha=0.7)
# ax.semilogy(np.arange(0,50,0.1), diff_478_norm, 'k', alpha=0.7)
ax.semilogy(np.arange(0,50,0.1), diff200_norm, 'k', alpha=0.7)
ax.set(xlabel='time', ylabel='Norm of difference', )
plt.show()

## Test adding a shadow into the domain

In [ ]:
# binary_snapshot [x,y,z,u]
def box(binary_snapshot, o, d):
    o = np.asarray(o)
    left = list(o.astype(int))
    right = list((o+d).astype(int))
    dim = len(left)
    
    def is_inside(coord):
        coord = list(coord)
        inside = []
        for i in range(dim):
            if left[i] < coord[i] < right[i]:
                inside.append(True)
        return len(inside) == dim
    
    snapshot_shape = binary_snapshot.shape
    grid = []
    for a in snapshot_shape:
        grid.append(np.arange(a))
    idx_group = np.meshgrid(*grid)
    idx = []
    for _idx in idx_group:
        idx.append(_idx.flatten())
    
    for coord in zip(*idx):
        if is_inside(coord):
            binary_snapshot[coord] = 0

    return binary_snapshot

In [ ]:
idx_group = np.meshgrid(np.arange(3), np.arange(3))
# print(idx_group[0].shape)
idx = []
for _idx in idx_group:
    # print(_idx.shape)
    # print(_idx.flatten().shape)
    idx.append(_idx.flatten())

for coord in zip(*idx):
    print(coord)

In [ ]:
binary_snapshot = np.zeros((100,100))
for _ in range(1000):
    i,j = np.random.randint(0,100,2)
    # print(i,j)
    binary_snapshot[i,j] = 1
binary_snapshot = box(binary_snapshot, [10,10], np.array([60,20]))

plt.figure()
plt.imshow(binary_snapshot,cmap='gray')
plt.show()

## Fourier modes

In [ ]:
# data200, data280
print(data200.shape, data280.shape)
# dim3_re34_k32_f4_dt01_grid64_478_t200-250.h5

In [ ]:
!ls /storage0/ym917/data/simulations/kolsol/raw_data/

In [ ]:
nref = 64 # number of physical grid
with h5py.File('/storage0/ym917/data/simulations/kolsol/raw_data/dim3_re34_k32_f4_dt01_478_t200-250.h5') as hf:
    nk = int(hf['nk'][()])
    nf = int(hf['nf'][()])
    dt_hat = float(hf['dt'][()])
    ndim = int(hf['ndim'][()])
    state_hat = np.array(hf.get('state_hat'))


In [ ]:
with h5py.File(data_path_original1) as hf:
    data200 = np.array(hf.get('state'))

In [ ]:
ishift = (nref - 2 * nk) // 2 # shift the 0 wavenumber to the middle, consistent with the solver
scaling = (nref / (2*nk)) ** 2
n_leading_dims = state_hat.ndim - (ndim + 1)
leading_dims = state_hat.shape[:n_leading_dims]
t_hat_aug = np.zeros(
    ([*leading_dims] + [nref for _ in range(ndim)] + [state_hat.shape[-1]]), dtype=np.complex128
) # empty aray
ishift_slice = [slice(ishift, ishift + 2*nk) for _ in range(ndim)]
t_hat_aug[tuple([...]) + tuple(ishift_slice) + tuple([slice(None)])] = state_hat # rifft this array would give the flow in physical space
print(ishift_slice, t_hat_aug.shape, state_hat.shape)
axs_lb, axs_ub = n_leading_dims, n_leading_dims + ndim

In [ ]:
## the grid the data is generated on
_kx = np.fft.fftshift(2*np.pi*np.fft.fftfreq(nk*2, d=2*np.pi/64))
_ky = np.fft.fftshift(2*np.pi*np.fft.fftfreq(nk*2, d=2*np.pi/64))
_kz = np.fft.fftshift(2*np.pi*np.fft.fftfreq(nk*2, d=2*np.pi/64))
_KX, _KY, _KZ = np.meshgrid(_kx, _ky, _kz, indexing="ij")
_Kmag = np.sqrt(_KX**2 + _KY**2, _KZ**2)
print(_Kmag.shape)

## The grid the data is interpolated onto
nx, ny, nz = t_hat_aug.shape[axs_lb:axs_ub]
kx = np.fft.fftshift(2*np.pi*np.fft.fftfreq(nx, d=2*np.pi/nx))
ky = np.fft.fftshift(2*np.pi*np.fft.fftfreq(ny, d=2*np.pi/ny))
kz = np.fft.fftshift(2*np.pi*np.fft.fftfreq(ny, d=2*np.pi/ny))
KX, KY, KZ = np.meshgrid(kx, ky, kz, indexing="ij")
Kmag = np.sqrt(KX**2 + KY**2, KZ**2)
print(Kmag.shape)

In [ ]:
## use energy (start from the highest energy)
percent_to_keep = 2.7/100.0
ke = np.sum(state_hat * np.conj(state_hat), axis=-1).real
ke_avg = np.mean(ke, axis=0)
num_to_keep = int(percent_to_keep*_Kmag.size)
print(f"Keeping {num_to_keep} out of {_Kmag.size} modes.")
flat_inds = np.argsort(ke_avg.ravel())[::-1]
_keep_inds = flat_inds[:num_to_keep]
_keep_idx = np.unravel_index(_keep_inds, _Kmag.shape)
keep_idx = (_keep_idx[0]+ishift, _keep_idx[1]+ishift)
# print(Kmag[*keep_idx].max(), Kmag[*keep_idx].size, Kmag[*keep_idx].flatten())

In [ ]:
# reconstruct
t_hat_trunc = np.zeros_like(t_hat_aug)
t_hat_trunc[:,*keep_idx,:] = t_hat_aug[:,*keep_idx,:]

phys_trunc = scaling * np.fft.irfftn(
    np.fft.ifftshift(t_hat_trunc, axes=range(axs_lb, axs_ub)),
    s=t_hat_aug.shape[slice(axs_lb, axs_ub)],
    axes=range(axs_lb, axs_ub)
)
print(f"Relative error of the fourier reconstruction is {100*float(relative_error(phys_trunc, data200)):.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7,4))

im0 = axes[0].imshow(
    np.abs(np.mean(t_hat_aug, axis=(0,4))[32,...]),
    extent=(-32,32,-32,32),
    origin='lower'
)

im1 = axes[1].imshow(
    np.abs(np.mean(t_hat_aug, axis=(0,4))[...,32]),
    extent=(-32,32,-32,32),
    origin='lower'
)

# Set grid spacing (change step as needed)
step = 4

for ax in axes:
    ax.set_xticks(np.arange(-32, 33, step))
    ax.set_yticks(np.arange(-32, 33, step))
    
    ax.grid(True, color='white', linewidth=0.5)
    
    # Make sure grid is drawn above image
    ax.set_axisbelow(False)

plt.show()

In [ ]:
print(f"{((5*2)**3 / (64**3))*100:.3f}% of grid grid points (uniformly spaced) needed to resolve up to Fourier modes k=5 in any direction.")